# pj-Flim movie agent — Colab PoC v0.1

Qwen2.5:7b/Ollama + LangChain + multilingual E5/FAISS. Chạy từ trên xuống trong notebook. Đây là agent tìm phim và dẫn nguồn có giới hạn, chưa tích hợp website hoặc huấn luyện model.

Notebook độc lập, không cần push GitHub. GPU/model trên Colab chưa được người tạo notebook chạy nghiệm thu thay bạn. Chọn dữ liệu/nguồn ở cell 1 trước khi Run all.

Nguồn: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Ollama Linux](https://docs.ollama.com/linux), [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama), [E5](https://huggingface.co/intfloat/multilingual-e5-small).


## 1. Chọn dữ liệu và nguồn

Mặc định dùng 4 phim giả do dự án tạo. Đổi USE_DEMO=False để nạp snapshot public của pj-Flim đã được phép xử lý cloud/embedding. Không đưa credential, lịch sử xem hoặc signed URL vào notebook. Source đi kèm có checksum; không cần GitHub đã cập nhật.


In [ ]:
# 1. Cấu hình + bộ source đi kèm: không cần clone/push GitHub.
USE_DEMO = True
REQUIRE_GPU = True
CONTEXT_TOKENS = 8192  # Nếu OOM: thử 4096, đo lại smoke và chất lượng.
ENABLE_OMDB = True  # Đọc OMDB_API_KEY từ Colab Secrets khi dùng catalog thật.
OMDB_MAPPINGS = {}  # UUID: {"imdb_id":"tt1375666","verified":True}; chỉ sau đối chiếu đúng tên/năm/loại.
ENABLE_TMDB = False
ENABLE_WEB_DISCOVERY = False  # Chỉ tìm URL; không đọc/copy bài review.
TMDB_MAPPINGS = {}  # UUID: {"kind":"movie" hoặc "tv","id":123,"verified":True}; chỉ sau khi đối chiếu đúng phim.

import base64, hashlib, json, platform, sys
from pathlib import Path
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("Mở notebook trong Google Colab Linux; notebook không cài Ollama trên PC.")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11+ required")
WORKSPACE = Path("/content/pj-flim-agent")
SOURCE = WORKSPACE / "source"
SOURCE.mkdir(parents=True, exist_ok=True)
encoded = "..."
payload = base64.b64decode(encoded)
if hashlib.sha256(payload).hexdigest() != "19a7917d0fe7b8871f4db4c19cddc5e36c46791aa53bbccf4cae35f7840c28cf":
    raise RuntimeError("BUNDLE_HASH_INVALID")
sources = json.loads(payload)
for relative_name, content in sources.items():
    target = (SOURCE / relative_name).resolve()
    if not target.is_relative_to(SOURCE.resolve()):
        raise RuntimeError("BUNDLE_PATH_INVALID")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
MODEL_CONFIG = json.loads((SOURCE / "model-versions.json").read_text())
print("Source bundle verified. Demo:", USE_DEMO, "OMDb:", ENABLE_OMDB, "TMDB:", ENABLE_TMDB, "Web:", ENABLE_WEB_DISCOVERY)


## 2. Kiểm runtime

Chọn Runtime → Change runtime type → GPU rồi chạy. Tài nguyên Colab không bảo đảm. Bản 7B cần đo VRAM/context thực, không coi file 4,7 GB là toàn bộ bộ nhớ.


In [ ]:
# Stdlib only: dependencies are installed in the next cell.
import platform, shutil, subprocess
HARDWARE = {"python": platform.python_version(), "system": platform.system(),
            "free_disk_gb": round(shutil.disk_usage("/content").free / 10**9, 2), "gpus": []}
try:
    report = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=10, check=True)
    HARDWARE["gpus"] = report.stdout.strip().splitlines()
except (OSError, subprocess.SubprocessError):
    pass
display(HARDWARE)
if REQUIRE_GPU and not HARDWARE["gpus"]:
    raise RuntimeError("GPU_NOT_ASSIGNED: chọn GPU trong Runtime và chạy lại. CPU thử nghiệm: REQUIRE_GPU=False.")
if HARDWARE["free_disk_gb"] < 10:
    raise RuntimeError("DISK_TOO_SMALL: cần khoảng trống cho Ollama/model/dependencies.")


## 3. Cài dependency

Cài các phiên bản trực tiếp đã pin, không nâng toàn bộ môi trường. Khi có yêu cầu restart kernel, restart session rồi chạy lại. Không có telemetry/tracing ứng dụng chủ động bật; tắt LangSmith tracing.


In [ ]:
import importlib.metadata
import subprocess
import sys
import os
from pathlib import Path
for key in ("LANGCHAIN_TRACING_V2", "LANGSMITH_TRACING"):
    os.environ[key] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
with (WORKSPACE / "dependency-install.log").open("w") as log:
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                                "-r", str(SOURCE / "requirements-colab.txt")], stdout=log, stderr=log)
if installed.returncode:
    raise RuntimeError("DEPENDENCY_INSTALL_FAILED: xem dependency-install.log, không chia sẻ key hoặc toàn bộ env.")
versions = {name: importlib.metadata.version(name) for name in
            ("langchain-core", "langchain-ollama", "pydantic", "numpy", "faiss-cpu", "sentence-transformers", "ddgs")}
display(versions)
import numpy
if numpy.__version__ != versions["numpy"]:
    raise RuntimeError("RESTART_KERNEL_REQUIRED: restart session rồi chạy notebook từ đầu để dùng đúng NumPy.")
# Model and RAG smoke tests below are required; do not infer compatibility from pip exit alone.


## 4. Ollama + Qwen

Cell này cài Ollama phiên bản đã chọn và tải Qwen2.5:7b nhiều GB vào runtime Google. Chỉ bind 127.0.0.1:11434, không tunnel hoặc public model. Chỉ tái sử dụng/dừng process notebook sở hữu. Runtime được reset có thể cần tải lại.


In [ ]:
import shutil
from pjflim_agent.catalog import AgentError
from pjflim_agent.runtime import start_ollama, pull_model
# Chỉ cài trong Colab. Kiểm/cài curl + zstd trước installer; không cần restart NumPy ở đây.
print("Đang kiểm tra/cài Ollama trong runtime Colab; có thể mất vài phút.")
try:
    display(start_ollama(WORKSPACE))
except AgentError as error:
    hints = {
        "OLLAMA_INSTALL_TOOLS_MISSING": "Thiếu curl/zstd và không tự cài được bằng apt-get quyền root.",
        "OLLAMA_DEPENDENCY_TIMEOUT": "apt-get quá thời gian; kiểm network runtime rồi chạy lại cell.",
        "OLLAMA_DEPENDENCY_INSTALL_FAILED": "apt-get lỗi; log riêng tại install-dependencies.log.",
        "OLLAMA_INSTALL_DOWNLOAD_FAILED": "Không tải được script HTTPS từ ollama.com; kiểm network/SSL.",
        "OLLAMA_INSTALL_DOWNLOAD_TIMEOUT": "Tải script quá thời gian; kiểm network rồi chạy lại cell.",
        "OLLAMA_INSTALL_DOWNLOAD_INVALID": "Script trả về rỗng hoặc quá giới hạn kích thước.",
        "OLLAMA_INSTALL_SCRIPT_FAILED": "Installer thoát lỗi; log riêng tại install.log.",
        "OLLAMA_INSTALL_TIMEOUT": "Installer vượt 15 phút; kiểm network/dung lượng runtime.",
        "OLLAMA_NOT_INSTALLED": "Installer kết thúc nhưng chưa tìm thấy executable ollama.",
    }
    display({"code": error.code, "hint": hints.get(error.code, "Xem hướng dẫn lỗi trong README agent."),
             "missing_install_tools": [t for t in ("curl", "zstd") if not shutil.which(t)]})
    raise RuntimeError(error.code) from None
print("Ollama đã khởi động; đang tải/kiểm model Qwen (nhiều GB).")
MODEL_MANIFEST = pull_model(WORKSPACE)
display(MODEL_MANIFEST)


## 5. Smoke model thật

Kiểm tiếng Việt, tool calling và JSON schema bằng model thực; nếu lỗi hãy dừng tại đây. Nhãn kiểm thử trong tài liệu repo chưa thay bằng chứng Colab. Không lưu chain-of-thought.


In [ ]:
import time
from pydantic import BaseModel, ConfigDict
from langchain_core.tools import StructuredTool
from pjflim_agent.runtime import make_model
llm = make_model(context_tokens=CONTEXT_TOKENS)
started = time.monotonic()
hello = llm.invoke("Trả lời tiếng Việt một câu ngắn: Bạn là trợ lý tìm phim.")
if not isinstance(hello.content, str) or not hello.content.strip():
    raise RuntimeError("MODEL_SMOKE_EMPTY")
print("Smoke tiếng Việt:", hello.content[:200])

class SmokeArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    title_id: str

smoke_tool = StructuredTool.from_function(
    func=lambda title_id: {"title_id": title_id},
    name="lookup_demo_title", description="Tra cứu phim demo bằng title_id.", args_schema=SmokeArgs)
probe = llm.bind_tools([smoke_tool]).invoke(
    "Hãy gọi lookup_demo_title với title_id chính xác là demo-sao-hoa. Không trả văn bản.")
if len(probe.tool_calls) != 1 or probe.tool_calls[0]["name"] != "lookup_demo_title" or probe.tool_calls[0]["args"] != {"title_id": "demo-sao-hoa"}:
    raise RuntimeError("MODEL_TOOL_SMOKE_FAILED: chưa thể coi model đã gọi tool đúng.")

class SmokeJson(BaseModel):
    ok: bool

typed = llm.with_structured_output(SmokeJson, method="json_schema").invoke("Trả JSON với ok=true.")
if not isinstance(typed, SmokeJson) or typed.ok is not True:
    raise RuntimeError("MODEL_JSON_SMOKE_FAILED")
MODEL_SMOKE = {"nonempty_text": True, "language_needs_human_review": True, "tool_call": True, "json_schema": True,
               "duration_seconds": round(time.monotonic()-started, 2)}
display(MODEL_SMOKE)


## 6. Nạp catalog

Demo mặc định không phải catalog thật. Với dữ liệu pj-Flim, chạy npm.cmd run agent:export-catalog trên PC khi API local đang chạy rồi chọn file .local/agent/catalog.snapshot.json. Snapshot hạn 24 giờ, có hash/schema và giới hạn 1.000 title. Đây là dữ liệu đủ điều kiện ở thời điểm xuất, không kiểm live.


In [ ]:
from pjflim_agent.catalog import Catalog, create_demo
from pjflim_agent.checkpoint import restore_checkpoint
CHECKPOINT_RESTORED = False
if USE_DEMO:
    catalog = create_demo(WORKSPACE / "catalog.snapshot.json")
else:
    from google.colab import files
    uploaded = files.upload()  # Chọn duy nhất catalog.snapshot.json hoặc checkpoint.zip trên PC.
    if len(uploaded) != 1:
        raise RuntimeError("SELECT_ONE_SNAPSHOT_OR_CHECKPOINT")
    name, content = next(iter(uploaded.items()))
    if name.lower().endswith(".zip"):
        if len(content) > 8_000_000:
            raise RuntimeError("CHECKPOINT_TOO_LARGE")
        archive_path = WORKSPACE / "input-checkpoint.zip"
        archive_path.write_bytes(content)
        restore_checkpoint(archive_path, WORKSPACE, expected_model_digest=MODEL_MANIFEST["digest"])
        CHECKPOINT_RESTORED = True
    elif name.lower().endswith(".json"):
        if len(content) > 5_000_000:
            raise RuntimeError("SNAPSHOT_TOO_LARGE")
        (WORKSPACE / "catalog.snapshot.json").write_bytes(content)
    else:
        raise RuntimeError("JSON_OR_ZIP_REQUIRED")
    catalog = Catalog.load(WORKSPACE / "catalog.snapshot.json")
display({"titles": len(catalog.by_id), "demo": catalog.snapshot.demo,
         "snapshot_at": catalog.snapshot.generated_at, "expires_at": catalog.snapshot.expires_at,
         "truncated": catalog.snapshot.truncated, "live_eligibility_checked": False})
if catalog.snapshot.demo:
    print("Đây là phim giả để thử agent; không phải phim đang có trên website.")


## 7. E5 + FAISS CPU

Pin embedding revision, dùng query:/passage:, normalize và exact inner product trên candidate đã lọc. Hợp keyword + semantic bằng RRF. Checkpoint lưu vectors.npy allow_pickle=False, kiểm hash/shape/version rồi dựng index; không load pickle hoặc FAISS binary từ nguồn tùy ý.


In [ ]:
from pjflim_agent.retrieval import HybridSearch
# E5 chạy CPU. Qwen dùng GPU nếu runtime/offload phù hợp.
search = HybridSearch(catalog, embedding_revision=MODEL_CONFIG["embeddingRevision"],
                      checkpoint_folder=WORKSPACE if CHECKPOINT_RESTORED else None)
search.save(WORKSPACE)
print("E5/FAISS sẵn sàng:", search.vectors.shape)
display([{"id": t.id, "name": t.name} for t in search.search("khoa học viễn tưởng", limit=3)])


## 8. Hỏi agent

Đổi QUESTION và chạy lại cell. CATALOG không gọi nguồn ngoài. HYBRID ưu tiên OMDb: thêm OMDB_API_KEY vào Colab Secrets và xác minh OMDB_MAPPINGS để lấy IMDb rating/Metascore. Tìm candidate bằng search_omdb_metadata; không tự duyệt mapping. TMDB vẫn tùy chọn với ENABLE_TMDB=True/TMDB_API_READ_TOKEN. ENABLE_WEB_DISCOVERY chỉ tìm URL theo tên phim; không đọc snippet/toàn văn. Có adapter OMDb lookup theo mapping; chưa bulk importer, Metacritic User Score, RT hoặc review summary. Demo chặn lookup ngoài.


In [ ]:
from pjflim_agent.agent import MovieAgent
from pjflim_agent.providers import Tmdb, ReviewDiscovery
from pjflim_agent.omdb import Omdb
from pjflim_agent.catalog import AgentError
from IPython.display import JSON

# Key chỉ đọc từ Secrets, không viết trực tiếp trong notebook hoặc output.
omdb_key, tmdb_token = "", ""
if (ENABLE_OMDB or ENABLE_TMDB) and not catalog.snapshot.demo:
    from google.colab import userdata
    if ENABLE_OMDB:
        try:
            omdb_key = userdata.get("OMDB_API_KEY") or ""
        except Exception:
            print("OMDB_NOT_CONFIGURED: thêm OMDB_API_KEY vào Colab Secrets, cấp quyền notebook.")
    if ENABLE_TMDB:
        try:
            tmdb_token = userdata.get("TMDB_API_READ_TOKEN") or ""
        except Exception:
            print("TMDB_NOT_CONFIGURED: thêm token vào Colab Secrets rồi cấp quyền notebook.")

# Giữ client OMDb qua các lần chạy cell8 để cache/quota không reset mỗi câu hỏi.
if "_omdb_client" not in globals() or not _omdb_client.matches_key(omdb_key):
    _omdb_client = Omdb(api_key=omdb_key, mappings=OMDB_MAPPINGS)
else:
    _omdb_client.mappings = OMDB_MAPPINGS
del omdb_key

tmdb = Tmdb(token=tmdb_token, mappings=TMDB_MAPPINGS)
discovery = ReviewDiscovery(enabled=ENABLE_WEB_DISCOVERY)
agent = MovieAgent(catalog, search, llm, tmdb=tmdb, discovery=discovery, omdb=_omdb_client)

# Demo: CATALOG, không gọi OMDb. Catalog thật: HYBRID để hỏi điểm.
QUESTION = "Tìm phim khoa học viễn tưởng nhẹ nhàng, tránh spoiler."
MODE = "CATALOG" if catalog.snapshot.demo else "HYBRID"
try:
    result = agent.ask(QUESTION, mode=MODE)
    display(JSON(result))
except AgentError as error:
    raise RuntimeError(error.code) from None

# Nếu chưa có IMDb ID: gọi _omdb_client.reset(); display(_omdb_client.search(catalog, TITLE_UUID))
# Đối chiếu candidate bằng tên/năm/loại, thêm vào OMDB_MAPPINGS ở cell1, chạy lại cell1 và cell8.


## 9. Nghiệm thu

Chạy contract tests bằng FAISS và ChatOllama thật với HTTP/model fixtures, rồi xuất báo cáo inference/retrieval của phiên này. Các nhóm live OMDb/TMDB/web chưa được đánh dấu đạt tự động. Dữ liệu nguồn chỉ là data; tool allowlist, schema, evidence registry và budget kiểm ngoài model.


In [ ]:
import json
import unittest
from datetime import datetime, timezone
suite = unittest.defaultTestLoader.discover(str(SOURCE / "tests"), pattern="test_*.py")
tests = unittest.TextTestRunner(verbosity=1).run(suite)
if not tests.wasSuccessful():
    raise RuntimeError("AGENT_CONTRACT_TESTS_FAILED")
REPORT = {"timestamp": datetime.now(timezone.utc).isoformat(), "hardware": HARDWARE,
          "dependencies": versions, "model": MODEL_MANIFEST, "model_smoke": MODEL_SMOKE, "context_tokens": CONTEXT_TOKENS,
          "embedding_model": MODEL_CONFIG["embeddingModel"], "embedding_revision": MODEL_CONFIG["embeddingRevision"],
          "embedding_shape": list(search.vectors.shape), "contract_tests": tests.testsRun,
          "demo": catalog.snapshot.demo, "agent_result_status": result["status"],
          "catalog_live_check": False, "omdb_live_test": False, "omdb_session_requests": _omdb_client.session_calls,
          "tmdb_live_test": False, "web_live_test": False}
(WORKSPACE / "runtime-report.json").write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
display(JSON(REPORT))
print("Nếu dùng snapshot thật: luôn kiểm lại publication/quyền qua backend trước khi đưa kết quả lên website.")


## 10. Checkpoint và kết thúc

Tải checkpoint về PC hoặc tự lưu nơi được phép. Không có prompt/API key/model weights trong checkpoint; source metadata vẫn cần quyền lưu. Sau reset: bootstrap, nạp snapshot/checkpoint chưa hết hạn, kiểm lại model/index. Đặt STOP_OLLAMA=True để dừng đúng process notebook tạo. Không dùng notebook free làm API/RabbitMQ worker 24/7.


In [ ]:
from pjflim_agent.checkpoint import save_checkpoint
from pjflim_agent.runtime import stop_ollama
archive_path = save_checkpoint(WORKSPACE)
print("Checkpoint:", archive_path, "— tải về trước khi runtime reset; không có key/prompt/model weights.")
DOWNLOAD_CHECKPOINT = False
STOP_OLLAMA = False
if DOWNLOAD_CHECKPOINT:
    from google.colab import files
    files.download(str(archive_path))
if STOP_OLLAMA:
    display(stop_ollama(WORKSPACE))
else:
    print("Chạy lại cell 8 để hỏi tiếp. Đặt STOP_OLLAMA=True ở cell này khi kết thúc.")
